# Bronze - Auto Loader (file notification mode) sobre S3

Ingesta **continua** de transacciones de tarjeta con Auto Loader — SLA de
latencia **< 15 minutos**.

## Modos de detección de archivos (doc: *File notification mode*, Oct 2026)

| Modo | Mecanismo | Latencia |
|---|---|---|
| Directory listing (default) | Polleo `LIST` periódico | minutos |
| File notification clásico | Cola SQS propia por stream (`cloudFiles.useNotifications`) | segundos |
| **File events (recomendado)**  | Cola `csms-*` gestionada por Databricks, ya suscrita a `transactions/` | segundos |

La external location `eurobank_s3_location` **tiene file events habilitados**
(test *File Events Read* 9/9) → basta con
`cloudFiles.useManagedFileEvents=true`. Requiere DBR 14.3+.

## Reglas clave
- **JSONL**: `format=json` asume un JSON por línea (el generador ya fue migrado)
- **Checkpoint** en volumen UC → estado *exactly-once* (RocksDB): si la query
  muere o reinicias, no procesa archivos dos veces
- **schemaLocation** guarda el esquema inferido (evolución de esquema)
- **Trigger `availableNow`**: serverless (tier gratuito) no admite triggers
  infinitos (`ProcessingTime`) — procesa todo lo disponible y termina; cada
  corrida retoma el checkpoint. En producción (tier de pago) se usa
  `.trigger(processingTime="30 seconds")` en un job — misma latencia de
  descubrimiento (file events), distinto disparo. SLA: < 15 min

Nota: La celda 1 hace `DROP TABLE` (reset inicial): la tabla de la ingesta anterior
se reemplaza por la ingerida por streaming desde JSONL.

In [ ]:
# 1. Estado limpio: volumen para checkpoints + tabla fresca
spark.sql("CREATE VOLUME IF NOT EXISTS eurobank_bronze.raw.checkpoints")
spark.sql("DROP TABLE IF EXISTS eurobank_bronze.raw.bronze_card_transactions")
print("Volumen listo y tabla reseteada")

In [ ]:
# 2. Fuente streaming: Auto Loader en modo file events (notificaciones, no polling)
SOURCE_PATH = "s3://eurobank-raw-oyvixj/transactions/"
CKPT = "/Volumes/eurobank_bronze/raw/checkpoints/auto_loader_tx"

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.useManagedFileEvents", "true")
    .option("cloudFiles.schemaLocation", f"{CKPT}/schema")
    .load(SOURCE_PATH)
)
print("Stream source listo (file notification via file events)")

In [ ]:
# 3. Destino streaming: tabla UC (datos en S3)
#    Serverless/tier gratuito no soporta triggers infinitos (ProcessingTime)
#    → AvailableNow: procesa todo lo disponible y termina (retoma checkpoint).
query = (
    raw_stream.writeStream
    .option("checkpointLocation", f"{CKPT}/checkpoint")
    .trigger(availableNow=True)
    .toTable("eurobank_bronze.raw.bronze_card_transactions")
)
query.awaitTermination()
print(f"Query completada: {query.id}")

In [ ]:
# 4. Verificación + medición de latencia
#    Tras subir un archivo nuevo a s3://.../transactions/: vuelve a correr las
#    celdas 2 y 3 (AvailableNow ingiere lo nuevo en segundos) y luego esta celda.
#    Los registros nuevos deben aparecer (SLA: < 15 min).
df = spark.table("eurobank_bronze.raw.bronze_card_transactions")
print(f"Registros en bronze: {df.count()}")
display(df.groupBy("status").count())